In [ ]:
# ================================================================================================
# Purchase Prediction Pipeline — FINAL (CV comparison, all models, extra features, correct fallback)
# ================================================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing   import OneHotEncoder, RobustScaler
from sklearn.impute          import SimpleImputer
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics         import (classification_report, roc_auc_score,
                                     f1_score, precision_recall_curve)
from scipy.sparse            import hstack
import xgboost as xgb


# ================================================================================================
# 0. Helper functions
# ================================================================================================

def undersample_train(X_train_fold, y_train_fold, pos_ratio=0.5, random_state=42):
    """Undersample majority class so that positive class makes up `pos_ratio` of the result."""
    pos_idx = y_train_fold[y_train_fold == 1].index
    neg_idx = y_train_fold[y_train_fold == 0].index
    n_pos = len(pos_idx)
    if n_pos == 0:
        return X_train_fold, y_train_fold
    n_neg = int(n_pos * (1.0 / pos_ratio - 1))
    n_neg = min(n_neg, len(neg_idx))
    rng = np.random.default_rng(random_state)
    sampled_neg = rng.choice(neg_idx, size=n_neg, replace=False)
    balanced_idx = np.concatenate([pos_idx, sampled_neg])
    rng.shuffle(balanced_idx)
    return X_train_fold.loc[balanced_idx], y_train_fold.loc[balanced_idx]


def tune_threshold(y_true, y_prob):
    """Find threshold that maximises F1 score."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores = 2 * (precision * recall) / (precision + recall)
        f1_scores = np.nan_to_num(f1_scores, nan=0.0)
    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    return best_thresh, f1_scores[best_idx]


def get_feature_importance(model, model_name, num_cols, cat_cols, encoder):
    all_names = list(num_cols) + list(encoder.get_feature_names_out(cat_cols))
    scores = model.feature_importances_
    imp = pd.Series(scores, index=all_names).sort_values(ascending=False)
    return imp


# ---- Expanding-window feature builder (with index-alignment fix) -----------------------------
def build_expanding_train_features(df, y_series):
    """Compute leak-free expanding-window features (per-pid order/click/basket rates,
    price std, target encodings for manufacturer/group/category) by walking through
    days in order. For each day, features reflect ONLY information from strictly
    earlier days, so no future labels leak into training.

    Returns (df_with_features, global_order_rate). The returned df is row-aligned
    with a fresh RangeIndex; the global rate is used as a fallback for unseen pids
    or categorical levels at test time.
    """
    df = df.copy()
    # Attach y as a column so it stays row-aligned through sort_values + reset_index.
    # Fixes IndexingError on newer pandas AND prevents silent misalignment between
    # sorted df rows and unsorted y values.
    df['__y__'] = y_series.reindex(df.index).to_numpy()
    df = df.sort_values('day').reset_index(drop=True)
    y_series = df.pop('__y__').astype(int)

    # Running state (cumulative through previous days only)
    pid_total  = {}; pid_order  = {}; pid_click = {}; pid_basket = {}; pid_prices = {}
    grp_total  = {}; grp_order  = {}
    man_total  = {}; man_order  = {}
    cat_total  = {}; cat_order  = {}
    global_orders = 0; global_total = 0

    # Initialise output columns
    for col in ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate', 'pid_price_std',
                'pid_seen_in_train', 'manufacturer_te', 'group_te', 'category_te']:
        df[col] = np.nan

    for day in sorted(df['day'].unique()):
        mask   = df['day'] == day
        day_df = df.loc[mask]
        y_day  = y_series.loc[mask]
        prev_rate = global_orders / global_total if global_total > 0 else 0.0

        # ----- Per-pid features -----
        for pid_val in day_df['pid'].unique():
            idx = day_df.index[day_df['pid'] == pid_val]
            total_pid = pid_total.get(pid_val, 0)
            if total_pid > 0:
                df.loc[idx, 'pid_order_rate']    = pid_order.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_click_rate']    = pid_click.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_basket_rate']   = pid_basket.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_seen_in_train'] = 1
            else:
                df.loc[idx, ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate']] = prev_rate
                df.loc[idx, 'pid_seen_in_train'] = 0
            past_prices = pid_prices.get(pid_val, [])
            df.loc[idx, 'pid_price_std'] = np.std(past_prices, ddof=0) if len(past_prices) >= 2 else 0.0

        # ----- Smoothed target encoding for categorical columns -----
        for col_orig, col_te, order_dict, total_dict in [
            ('manufacturer', 'manufacturer_te', man_order, man_total),
            ('group',        'group_te',        grp_order, grp_total),
            ('category',     'category_te',     cat_order, cat_total),
        ]:
            for val in day_df[col_orig].unique():
                idx_val = day_df.index[day_df[col_orig] == val]
                n_ord = order_dict.get(val, 0)
                n_tot = total_dict.get(val, 0)
                smoothing = 10
                te = ((n_ord + prev_rate * smoothing) / (n_tot + smoothing)) if (n_tot + smoothing) > 0 else prev_rate
                df.loc[idx_val, col_te] = te

        # ----- Update state with today's data (only AFTER today's features are written) -----
        pid_day_counts = day_df.groupby('pid').size()
        pid_day_orders = y_day.groupby(day_df['pid']).sum()
        for pid_val in pid_day_counts.index:
            pid_total[pid_val] = pid_total.get(pid_val, 0) + pid_day_counts[pid_val]
            pid_order[pid_val] = pid_order.get(pid_val, 0) + pid_day_orders.get(pid_val, 0)

        for beh_col, cum_dict in [('click', pid_click), ('basket', pid_basket)]:
            day_sum = day_df.groupby('pid')[beh_col].sum()
            for pid_val in day_sum.index:
                cum_dict[pid_val] = cum_dict.get(pid_val, 0) + day_sum[pid_val]

        for pid_val, prices in day_df.groupby('pid')['price'].apply(list).items():
            if pid_val not in pid_prices:
                pid_prices[pid_val] = []
            pid_prices[pid_val].extend(prices)

        for col_orig, order_dict, total_dict in [
            ('manufacturer', man_order, man_total),
            ('group',        grp_order, grp_total),
            ('category',     cat_order, cat_total),
        ]:
            day_grp = day_df.groupby(col_orig)
            day_cnt = day_grp.size()
            day_ord = y_day.groupby(day_df[col_orig]).sum()
            for val in day_cnt.index:
                total_dict[val] = total_dict.get(val, 0) + day_cnt[val]
                order_dict[val] = order_dict.get(val, 0) + day_ord.get(val, 0)

        global_total  += len(day_df)
        global_orders += y_day.sum()

    final_global_rate = global_orders / global_total if global_total > 0 else 0.0
    return df, y_series, final_global_rate


# ================================================================================================
# 1. Load Data
# ================================================================================================
train = pd.read_csv('/Users/annageiser/Documents/GITHUB/analytics-project/data/raw/train.csv', sep='|')
items = pd.read_csv('/Users/annageiser/Documents/GITHUB/analytics-project/data/raw/items.csv', sep='|')
df = train.merge(items, on='pid', how='left', validate='m:1')
print("Data loaded and merged.")

# ================================================================================================
# 2. Sort and sample (set SAMPLING = 1.0 for final run)
# ================================================================================================
SAMPLING = 1   # use 0.5 for speed during development, 1.0 for final results
df = df.sort_values(['day', 'lineID']).reset_index(drop=True)
if SAMPLING < 1.0:
    df = (df.groupby('day', group_keys=False)
            .sample(frac=SAMPLING, random_state=42)
            .sort_values(['day', 'lineID'])
            .reset_index(drop=True))
print(f"Using {len(df):,} rows ({SAMPLING*100:.0f}% stratified-by-day sample).")

# ================================================================================================
# 3. Feature Engineering (including new temporal features)
# ================================================================================================

# --- 3a. Basic features (unchanged) ---
df['priceRatio'] = (df['price'] / df['rrp'].replace(0, np.nan)).fillna(1.0)
df['missingCompetitorPrice'] = df['competitorPrice'].isnull().astype(int)

df['weekDay_raw'] = (df['day'] % 7).replace({0: 7})
df['weekDay_sin'] = np.sin(2 * np.pi * df['weekDay_raw'] / 7)
df['weekDay_cos'] = np.cos(2 * np.pi * df['weekDay_raw'] / 7)
df.drop(columns=['weekDay_raw'], inplace=True)

df['priceVsCompetitor'] = (
    df['price'] / df['competitorPrice'].replace(0, np.nan)
).fillna(1.0)
df['priceDiscount'] = (df['rrp'] - df['price']).clip(lower=0)

df['adFlag_x_priceRatio'] = df['adFlag'] * df['priceRatio']
df['avail_x_priceRatio']  = df['availability'] * df['priceRatio']
df['regulated_generic']   = df['salesIndex'] * df['genericProduct']

# --- 3b. NEW: Time since last promotion (adFlag lag) ---
# For each product, compute days since the most recent day where adFlag==1
df = df.sort_values(['pid','day'])
df['last_promo_day'] = df.groupby('pid')['day'].shift(1)
df['days_since_last_promo'] = (df['day'] - df['last_promo_day']).fillna(999).astype(int)
# Only count days with adFlag==1 as promotions
promo_mask = df['adFlag'] == 1
df['last_promo_day'] = np.where(promo_mask, df['day'], np.nan)
# Forward fill the last promo day within each pid, then compute difference
df['last_promo_day'] = df.groupby('pid')['last_promo_day'].ffill()
df['days_since_promo'] = (df['day'] - df['last_promo_day']).fillna(999).astype(int)
df.drop(columns=['last_promo_day', 'days_since_last_promo'], inplace=True)

# --- 3c. NEW: Product age (days since first seen) ---
df['first_seen'] = df.groupby('pid')['day'].transform('min')
df['product_age_days'] = df['day'] - df['first_seen']
df.drop(columns=['first_seen'], inplace=True)

# --- 3d. NEW: Competitor price dynamics (rolling average and trend) ---
# 7‑day rolling average and linear trend (slope) of competitorPrice per product
df = df.sort_values(['pid','day'])
df['competitorPrice_7day_avg'] = df.groupby('pid')['competitorPrice'].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
# Simple trend: slope of best‑fit line over last 7 days
def rolling_slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x))
    slope = np.polyfit(t, x, 1)[0]  # linear coefficient
    return slope
df['competitorPrice_trend'] = df.groupby('pid')['competitorPrice'].transform(
    lambda x: x.rolling(7, min_periods=2).apply(rolling_slope, raw=True)
)

# ================================================================================================
# 4. Train / Test Split (time‑based, 80/20)
# ================================================================================================
X = df.drop(columns=['order'])
y = df['order']
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()
print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test positive rate: {y_test.mean():.3f}")

# ================================================================================================
# 5. Feature lists (updated with new features)
# ================================================================================================
NUM_FEATURES = [
    'competitorPrice', 'priceRatio', 'priceVsCompetitor', 'priceDiscount',
    'missingCompetitorPrice', 'weekDay_sin', 'weekDay_cos', 'pid_order_rate',
    'pid_click_rate', 'pid_basket_rate', 'manufacturer_te', 'group_te',
    'category_te', 'adFlag_x_priceRatio', 'avail_x_priceRatio',
    'regulated_generic', 'pid_price_std', 'pid_seen_in_train',
    'days_since_promo',          # NEW
    'product_age_days',          # NEW
    'competitorPrice_7day_avg',  # NEW
    'competitorPrice_trend'      # NEW
]
CAT_FEATURES = [
    'adFlag', 'availability', 'content', 'unit', 'pharmForm',
    'genericProduct', 'salesIndex'
]
DROP_COLS = [
    # Genuine target leakage (mutually exclusive with `order` per the project brief)
    'click', 'basket', 'revenue',
    # Encoded into engineered features (priceRatio, priceDiscount, priceVsCompetitor)
    'price', 'rrp',
    # Identifiers / not features
    'lineID', 'day', 'pid',
    # ~94% missing → unusable. Note correct spelling matches column in items.csv
    'campainIndex',
]
TE_COLS = [
    ('manufacturer', 'manufacturer_te'),
    ('group',        'group_te'),
    ('category',     'category_te'),
]


# ================================================================================================
# 6. Cross‑validation experiment (rolling vs expanding) — with all three models
# ================================================================================================
def run_cv_experiment(cv_mode='rolling', train_windows=[7,14]):
    all_results = []
    unique_days = sorted(X_train['day'].unique())
    for window_size in (train_windows if cv_mode=='rolling' else [None]):
        if cv_mode == 'rolling':
            cv_label = f"rolling_{window_size}d"
            print(f"\n{'='*70}\nRolling CV with TRAIN_WINDOW={window_size}\n{'='*70}")
        else:
            cv_label = "expanding"
            print(f"\n{'='*70}\nExpanding CV (TimeSeriesSplit)\n{'='*70}")

        if cv_mode == 'rolling':
            folds = []
            for i in range(window_size, len(unique_days)-1):
                folds.append((unique_days[i-window_size:i], unique_days[i:i+1]))
        else:
            n_splits = min(len(unique_days)-2, 20)
            tscv = TimeSeriesSplit(n_splits=n_splits)
            folds = [(unique_days[train_idx], unique_days[test_idx])
                     for train_idx, test_idx in tscv.split(unique_days)]

        for fold_i, (train_days, val_days) in enumerate(folds):
            if fold_i > 40:  # limit for speed
                break
            train_idx = X_train[X_train['day'].isin(train_days)].index
            val_idx   = X_train[X_train['day'].isin(val_days)].index
            X_fold_train = X_train.loc[train_idx].copy()
            X_fold_val   = X_train.loc[val_idx].copy()
            y_fold_train = y_train.loc[train_idx].copy()
            y_fold_val   = y_train.loc[val_idx].copy()
            if len(X_fold_train) < 100 or len(X_fold_val) < 50:
                continue

            # ----- Temporal features (anti‑leakage) -----
            fold_labels = pd.concat([X_fold_train[['pid']], y_fold_train], axis=1)
            pid_rate = fold_labels.groupby('pid')['order'].mean().rename('pid_order_rate')
            X_fold_train = X_fold_train.join(pid_rate, on='pid')
            X_fold_val   = X_fold_val.join(pid_rate, on='pid')

            for beh_col, rate_name in [('click','pid_click_rate'),('basket','pid_basket_rate')]:
                beh_rate = X_fold_train.groupby('pid')[beh_col].mean().rename(rate_name)
                X_fold_train = X_fold_train.join(beh_rate, on='pid')
                X_fold_val   = X_fold_val.join(beh_rate, on='pid')

            price_stats = X_fold_train.groupby('pid')['price'].agg(['std']).rename(columns={'std':'pid_price_std'})
            X_fold_train = X_fold_train.join(price_stats, on='pid')
            X_fold_val   = X_fold_val.join(price_stats, on='pid')

            train_pids = set(X_fold_train['pid'].unique())
            X_fold_train['pid_seen_in_train'] = X_fold_train['pid'].isin(train_pids).astype(int)
            X_fold_val['pid_seen_in_train']   = X_fold_val['pid'].isin(train_pids).astype(int)

            global_rate = y_fold_train.mean()
            for col, te_col in TE_COLS:
                te_temp = pd.concat([X_fold_train[[col]], y_fold_train], axis=1)
                group_stats = te_temp.groupby(col)['order'].agg(['mean','count'])
                smooth = ((group_stats['mean']*group_stats['count'] + global_rate*10) /
                          (group_stats['count']+10)).to_dict()
                X_fold_train[te_col] = X_fold_train[col].map(smooth).fillna(global_rate)
                X_fold_val[te_col]   = X_fold_val[col].map(smooth).fillna(global_rate)

            for col, _ in TE_COLS:
                X_fold_train.drop(columns=[col], errors='ignore', inplace=True)
                X_fold_val.drop(  columns=[col], errors='ignore', inplace=True)
            X_fold_train.drop(columns=LEAKAGE_COLS, errors='ignore', inplace=True)
            X_fold_val.drop(  columns=LEAKAGE_COLS, errors='ignore', inplace=True)

            # ----- Correlation filter -----
            active_num = [f for f in NUM_FEATURES if f in X_fold_train.columns]
            corr = X_fold_train[active_num].corr().abs()
            upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            drop = [col for col in upper.columns if any(upper[col] > 0.95)]
            X_fold_train.drop(columns=drop, errors='ignore', inplace=True)
            X_fold_val.drop(  columns=drop, errors='ignore', inplace=True)
            active_num = [f for f in active_num if f not in drop]
            active_cat = [f for f in CAT_FEATURES if f in X_fold_train.columns]

            # ----- Impute & cast -----
            fill_vals = {col: 'Missing' for col in active_cat}
            X_fold_train.fillna(fill_vals, inplace=True)
            X_fold_val.fillna(  fill_vals, inplace=True)
            X_fold_train[active_cat] = X_fold_train[active_cat].astype(str)
            X_fold_val[  active_cat] = X_fold_val[  active_cat].astype(str)

            # ----- Undersample training fold -----
            X_fold_train, y_fold_train = undersample_train(X_fold_train, y_fold_train)

            # ----- Encode & scale -----
            imputer = SimpleImputer(strategy='median')
            scaler  = RobustScaler()
            encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
            X_tr_num = scaler.fit_transform(imputer.fit_transform(X_fold_train[active_num]))
            X_val_num = scaler.transform(imputer.transform(X_fold_val[active_num]))
            X_tr_cat = encoder.fit_transform(X_fold_train[active_cat])
            X_val_cat = encoder.transform(X_fold_val[active_cat])
            X_tr_fold = hstack([X_tr_num, X_tr_cat])
            X_val_fold = hstack([X_val_num, X_val_cat])

            # ----- Train & evaluate all three models -----
            spw = (y_fold_train==0).sum()/(y_fold_train==1).sum() if y_fold_train.sum()>0 else 1
            models = {
                'DecisionTree': DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
                'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10,
                                                       class_weight='balanced_subsample',
                                                       n_jobs=-1, random_state=42),
                'XGBoost': xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                             scale_pos_weight=spw, eval_metric='auc',
                                             use_label_encoder=False, random_state=42, verbosity=0)
            }
            for name, model in models.items():
                model.fit(X_tr_fold, y_fold_train)
                y_prob = model.predict_proba(X_val_fold)[:,1]
                thresh, best_f1 = tune_threshold(y_fold_val, y_prob)
                y_pred = (y_prob >= thresh).astype(int)
                auc = roc_auc_score(y_fold_val, y_prob)
                f1  = f1_score(y_fold_val, y_pred, zero_division=0)
                all_results.append({
                    'cv_mode': cv_label, 'fold': fold_i, 'model': name,
                    'auc': auc, 'f1': f1, 'best_threshold': thresh,
                    'val_day': val_days[-1], 'train_size': len(X_fold_train),
                    'val_size': len(X_fold_val)
                })
            if fold_i % 10 == 0:
                print(f"  Fold {fold_i:3d} done.")
    return pd.DataFrame(all_results)

# Run CV (comment out for quick final model test if time is short)
# df_cv = run_cv_experiment(cv_mode='rolling', train_windows=[7,14])
# df_cv.to_csv('cv_results.csv', index=False)
# print("\nCV summary:")
# print(df_cv.groupby(['cv_mode','model'])[['auc','f1']].mean().round(4))


# ================================================================================================
# 7. Final model — expanding‑window features, proper validation, HPO, all models evaluated
# ================================================================================================
print("\n=== Final Model — Expanding-Window Features ===")


# ---- 7a. Day-aware CV splitter -------------------------------------------------------
def make_day_folds(day_array, n_splits=3):
    """Build (train_idx, val_idx) folds that grow forward in time, by day.

    Returns positional indices into the array passed to .fit(), so the day_array
    must come from the same row order as the matrix used for HPO.
    """
    unique_days = np.sort(np.unique(day_array))
    n_days = len(unique_days)
    fold_size = max(1, n_days // (n_splits + 1))
    folds = []
    for i in range(n_splits):
        train_end = (i + 1) * fold_size
        val_end   = min(train_end + fold_size, n_days)
        train_set = set(unique_days[:train_end])
        val_set   = set(unique_days[train_end:val_end])
        train_idx = np.where(np.isin(day_array, list(train_set)))[0]
        val_idx   = np.where(np.isin(day_array, list(val_set)))[0]
        if len(train_idx) > 0 and len(val_idx) > 0:
            folds.append((train_idx, val_idx))
    return folds


# ---- 7b. Cold-start augmentation -----------------------------------------------------
def augment_cold_start(X, global_rate, frac=0.05, random_state=42):
    """Randomly mask `frac` of training rows so they look like a never-seen pid:
    pid_seen_in_train=0 and pid_*_rate features reset to the global fallback.

    The expanding-window builder already produces some 0-rows on each pid's first
    appearance, but those rarely carry meaningful pid history elsewhere in the
    batch. This augmentation gives the model paired examples (same row, but with
    history hidden), which is the actual cold-start case at test time.
    """
    rng = np.random.default_rng(random_state)
    n_mask = int(len(X) * frac)
    if n_mask == 0:
        return X
    mask_idx = rng.choice(X.index, size=n_mask, replace=False)
    X = X.copy()
    for col in ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate']:
        if col in X.columns:
            X.loc[mask_idx, col] = global_rate
    if 'pid_seen_in_train' in X.columns:
        X.loc[mask_idx, 'pid_seen_in_train'] = 0
    if 'pid_price_std' in X.columns:
        X.loc[mask_idx, 'pid_price_std'] = 0.0
    return X


# ---- 7c. Test-time feature lookup ---------
def build_test_features(test_df, train_df_with_features, global_fallback_rate):
    pid_cols = ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate',
                'pid_price_std', 'pid_seen_in_train']
    train_sorted = train_df_with_features.sort_values('day')

    # Forward-fill within each pid, then take the last row → latest non-null per column
    filled = train_sorted.copy()
    filled[pid_cols] = filled.groupby('pid')[pid_cols].ffill()
    pid_feats = filled.groupby('pid')[pid_cols].last()

    test_df = test_df.join(pid_feats, on='pid')
    for col in ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate']:
        test_df[col] = test_df[col].fillna(global_fallback_rate)
    test_df['pid_price_std']     = test_df['pid_price_std'].fillna(0.0)
    test_df['pid_seen_in_train'] = test_df['pid_seen_in_train'].fillna(0).astype(int)

    # Same logic for the target-encoded categorical columns
    for col_orig, col_te in TE_COLS:
        te_filled = train_sorted.copy()
        te_filled[col_te] = te_filled.groupby(col_orig)[col_te].ffill()
        te_map = te_filled.groupby(col_orig)[col_te].last()
        test_df[col_te] = test_df[col_orig].map(te_map).fillna(global_fallback_rate)

    return test_df


# ---- 7d. Build expanding-window features on training data ------------------------------------
X_train_expanded, y_train_expanded, global_rate = build_expanding_train_features(X_train.copy(), y_train)
print(f"Global order rate (fallback for unseen pids/levels): {global_rate:.4f}")


# ---- 7e. Split training data into dev (HPO + fit) and cal (threshold tuning) -----------------
unique_days = sorted(X_train_expanded['day'].unique())
cal_size = max(1, int(len(unique_days) * 0.10))
cal_days = unique_days[-cal_size:]
dev_days = [d for d in unique_days if d not in cal_days]
print(f"Days: {len(unique_days)} total | dev: {len(dev_days)} | cal: {len(cal_days)}")

X_dev = (X_train_expanded[X_train_expanded['day'].isin(dev_days)]
         .sort_values('day').copy())
y_dev = y_train_expanded.loc[X_dev.index].copy()
X_cal = (X_train_expanded[X_train_expanded['day'].isin(cal_days)]
         .sort_values('day').copy())
y_cal = y_train_expanded.loc[X_cal.index].copy()


# ---- 7f. Build test features ---------------------------------------------
X_test_expanded = build_test_features(X_test.copy(), X_train_expanded, global_rate)


# ---- 7g. Cold-start augmentation on dev only -----------------------------------------
X_dev_aug = augment_cold_start(X_dev, global_rate=global_rate, frac=0.05, random_state=42)


# ---- 7h. Capture day vector for CV folds, then drop non-feature columns ----------------------
# X_dev_aug preserves the day-sorted row order, which becomes the row order of X_dev_proc.
dev_days_array = X_dev_aug['day'].values

for d in [X_dev_aug, X_cal, X_test_expanded]:
    d.drop(columns=[c for c, _ in TE_COLS], errors='ignore', inplace=True)
    d.drop(columns=DROP_COLS, errors='ignore', inplace=True)


# ---- 7i. Feature lists & correlation filter (fit on dev only) --------------------------------
active_num = [f for f in NUM_FEATURES if f in X_dev_aug.columns]
corr  = X_dev_aug[active_num].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = [c for c in upper.columns if any(upper[c] > 0.95)]
if high_corr:
    print(f"Dropping correlated features (>0.95): {high_corr}")
for d in [X_dev_aug, X_cal, X_test_expanded]:
    d.drop(columns=high_corr, errors='ignore', inplace=True)
active_num = [f for f in active_num if f not in high_corr]
active_cat = [f for f in CAT_FEATURES if f in X_dev_aug.columns]


# ---- 7j. Impute & cast categoricals ----------------------------------------------------------
fill = {col: 'Missing' for col in active_cat}
for d in [X_dev_aug, X_cal, X_test_expanded]:
    d.fillna(fill, inplace=True)
    d[active_cat] = d[active_cat].astype(str)


# ---- 7k. Preprocessing — fit on dev only -----------------
imputer = SimpleImputer(strategy='median')
scaler  = RobustScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

X_dev_num  = scaler.fit_transform(imputer.fit_transform(X_dev_aug[active_num]))
X_dev_cat  = encoder.fit_transform(X_dev_aug[active_cat])
X_dev_proc = hstack([X_dev_num, X_dev_cat]).tocsr()

X_cal_num  = scaler.transform(imputer.transform(X_cal[active_num]))
X_cal_cat  = encoder.transform(X_cal[active_cat])
X_cal_proc = hstack([X_cal_num, X_cal_cat]).tocsr()

X_tst_num  = scaler.transform(imputer.transform(X_test_expanded[active_num]))
X_tst_cat  = encoder.transform(X_test_expanded[active_cat])
X_tst_proc = hstack([X_tst_num, X_tst_cat]).tocsr()


# ---- 7l. HPO with day-aware CV (fixes 1, 5, 6) -----------------------------------------------
day_folds = make_day_folds(dev_days_array, n_splits=3)
print(f"Built {len(day_folds)} day-aware CV folds for HPO.")

# Fix 6: no undersampling. scale_pos_weight reflects the natural ~1:2.9 imbalance.
spw = (y_dev == 0).sum() / max((y_dev == 1).sum(), 1)
print(f"scale_pos_weight (XGBoost) on natural distribution: {spw:.3f}")

dt_param_grid = {
    'max_depth': [5, 8, 12, None],
    'min_samples_leaf': [10, 50, 100],
}
rf_param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [8, 10, 12],
    'min_samples_leaf': [20, 50],
}
xgb_param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.7, 0.8],
    'min_child_weight': [3, 5],
}

# n_jobs=1 on the inner estimators avoids nested parallelism with the outer search.
model_specs = [
    ('DecisionTree',
     DecisionTreeClassifier(class_weight='balanced', random_state=42),
     dt_param_grid),
    ('RandomForest',
     RandomForestClassifier(class_weight='balanced_subsample',
                            n_jobs=1, random_state=42),
     rf_param_grid),
    ('XGBoost',
     xgb.XGBClassifier(scale_pos_weight=spw, eval_metric='auc',
                       random_state=42, verbosity=0, n_jobs=1),
     xgb_param_grid),
]

searches = {}   # fix 1: per-model HPO storage
for name, base_model, param_grid in model_specs:
    print(f"\nHPO for {name} ...")
    search = RandomizedSearchCV(
        base_model, param_grid, cv=day_folds,
        scoring='roc_auc', n_iter=10, random_state=42, n_jobs=-1, refit=True,
    )
    search.fit(X_dev_proc, y_dev.values)
    searches[name] = search
    print(f"  Best params:  {search.best_params_}")
    print(f"  Best CV AUC:  {search.best_score_:.4f}")


# ---- 7m. Tune threshold on cal & evaluate on test (fixes 2, 7) -------------------------------
# Critical: the SAME model used for threshold tuning is used for test prediction.
# No retraining on dev+cal, so probability calibration is preserved end-to-end.
print("\n=== Final Test Set Results ===")
test_results = []
for name, search in searches.items():
    model = search.best_estimator_   # already refit on full X_dev_proc by RandomizedSearchCV

    # Threshold from cal slice
    cal_prob = model.predict_proba(X_cal_proc)[:, 1]
    thr, cal_f1 = tune_threshold(y_cal.values, cal_prob)

    # Test eval with same model + same threshold
    test_prob = model.predict_proba(X_tst_proc)[:, 1]
    test_pred = (test_prob >= thr).astype(int)
    test_auc  = roc_auc_score(y_test, test_prob)
    test_f1   = f1_score(y_test, test_pred, zero_division=0)

    print(f"\n{name}: cal-tuned threshold = {thr:.3f}  (cal F1 = {cal_f1:.4f})")
    print(f"  Test AUC = {test_auc:.4f}  |  Test F1 = {test_f1:.4f}")
    print(classification_report(y_test, test_pred, target_names=['no order', 'order']))

    if hasattr(model, 'feature_importances_'):
        imp = get_feature_importance(model, name, active_num, active_cat, encoder)
        print(f"  Top 10 features ({name}):")
        print(imp.head(10).to_string())

    test_results.append({
        'model':         name,
        'best_params':   str(search.best_params_),
        'cv_auc':        search.best_score_,
        'cal_threshold': thr,
        'cal_f1':        cal_f1,
        'test_auc':      test_auc,
        'test_f1':       test_f1,
    })

pd.DataFrame(test_results).to_csv('final_test_results.csv', index=False)
print("\nPipeline completed successfully.")

Data loaded and merged.
Using 275,601 rows (10% stratified-by-day sample).
Train size: 220,480  |  Test size: 55,121
Train positive rate: 0.264  |  Test positive rate: 0.223

=== Final Model — Expanding-Window Features ===
Global order rate (fallback for unseen pids/levels): 0.2638
Days: 92 total | dev: 83 | cal: 9
Dropping correlated features (>0.95): ['days_since_promo', 'competitorPrice_7day_avg']
Built 3 day-aware CV folds for HPO.
scale_pos_weight (XGBoost) on natural distribution: 2.780

HPO for DecisionTree ...
  Best params:  {'min_samples_leaf': 100, 'max_depth': 8}
  Best CV AUC:  0.6712

HPO for RandomForest ...
  Best params:  {'n_estimators': 150, 'min_samples_leaf': 20, 'max_depth': 12}
  Best CV AUC:  0.6879

HPO for XGBoost ...
  Best params:  {'subsample': 0.7, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.05}
  Best CV AUC:  0.6921

=== Final Test Set Results ===

DecisionTree: cal-tuned threshold = 0.450  (cal F1 = 0.4868)
  Test AUC 